Importing requried libaries

In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/ValidationFramework

In [0]:


start_time = datetime.now()

print(start_time)

Handling Schema **Validations**

In [0]:


healthcare_schema =F.StructType([
    StructField("Id", StringType(), True),
    StructField("Birthdate", DateType(), True),
    StructField("Deathdate", DateType(), True),
    StructField("Ssn", StringType(), True),
    StructField("Drivers", StringType(), True),
    StructField("Passport", StringType(), True),
    StructField("Prefix", StringType(), True),
    StructField("First", StringType(), True),
    StructField("Middle", StringType(), True),
    StructField("Last", StringType(), True),
    StructField("Suffix", StringType(), True),
    StructField("Maiden", StringType(), True),
    StructField("Marital", StringType(), True),
    StructField("Race", StringType(), True),
    StructField("Ethnicity", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Birthplace", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Fips", IntegerType(), True),
    StructField("Zip", StringType(), True),
    StructField("Lattitude", DoubleType(), True),
    StructField("Longtitude", DoubleType(), True),
    StructField("Healthcare_Expenses", DoubleType(), True),
    StructField("Healthcare_Coverage", DoubleType(), True),
    StructField("Income", DoubleType(), True)
])

Reading Source File

In [0]:


df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(healthcare_schema)
    .load("/Volumes/helathcare_bronze/helathcare_bronze/healthcare/patients.csv")

    .withColumn("Readtimestamp", F.current_timestamp())

    .withColumn("Filename", F.col("_metadata.file_name"))

    .withColumn("file_size", F.col("_metadata.file_size"))
)

display(df_raw)

Creating Bronze Table

In [0]:

# Drop table if exists to avoid metadata mismatch
#spark.sql(f"DROP TABLE IF EXISTS helathcare_bronze.{bronze_schema}.PatientData")

df_raw.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"helathcare_bronze.{bronze_schema}.PatientData")

In [0]:
%sql
-- SELECT distinct COUNTY,FIPS FROM `helathcare_bronze`.`helathcare_bronze`.`PatientData` where FIPS is not null and COUNTY in ('Middlesex County','Plymouth County','Bristol County','Worcester County','Essex County','Hampden County','Barnstable County','Norfolk County','Franklin County','Berkshire County');

SELECT distinct * FROM `helathcare_bronze`.`helathcare_bronze`.`PatientData`


In [0]:
%sql
--to check version history
-- DESCRIBE HISTORY 
-- helathcare_bronze.helathcare_bronze.PatientData
-- -- if want to read version data
-- SELECT *
-- FROM helathcare_bronze.helathcare_bronze.PatientData
-- VERSION AS OF 1
-- %sql
-- to read current latest version
-- DESCRIBE DETAIL
-- helathcare_bronze.helathcare_bronze.PatientData
--if you want to restore to a specific version
-- ---RESTORE TABLE 
-- helathcare_bronze.helathcare_bronze.PatientData
-- TO VERSION AS OF 1

-- In Delta Lake, we can use DESCRIBE HISTORY to check table versions and transaction history. Delta maintains versioned data through the _delta_log transaction log, which enables time travel, auditing, rollback, and recovery capabilities.


Silver Validation


In [0]:
#Checking for null values in the bronze table
silver_df = spark.table("helathcare_bronze.helathcare_bronze.PatientData")


Validation

In [0]:
validation_result = run_validations(
    silver_df,
    ["Id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

Handling Nulls


In [0]:
#--in DRIVERS have null values,so fill with sequence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
from pyspark.sql.window import Window

start_num = 99999871

window_spec = Window.orderBy(F.monotonically_increasing_id())

silver_df = silver_df.withColumn(
    "seq_num",
    F.when(
        F.col("Drivers").isNull(),
        F.row_number().over(window_spec) + start_num
    )
)

silver_df = silver_df.withColumn(
    "Drivers",
    F.when(
        F.col("Drivers").isNull(),
        F.concat(F.lit("S"), F.col("seq_num"))
    ).otherwise(F.col("Drivers"))
).drop("seq_num")

display(silver_df)

In [0]:
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
silver_df = silver_df.withColumn(
    "Prefix",

    F.when(
        (F.col("Prefix").isNull()) &
        (F.col("Gender") == "M"),
        "Mr."
    ).when(
        (F.col("Prefix").isNull()) &
        (F.col("Gender") == "F"),
        "Mrs."
    ).otherwise(F.col("Prefix"))
)
display(silver_df)

In [0]:
#---FIPS have nulls values , need to fill with county name having filps code
# Lookup table
fips_lookup = (
    silver_df.filter(F.col("Fips").isNotNull())
    .select(
        F.col("Country").alias("lkp_county"),
        F.col("Fips").alias("lkp_fips")
    )
    .dropDuplicates(["lkp_county"])
)

# Join and update
silver_df = (
    silver_df.alias("main")
    .join(
        fips_lookup,
        F.col("main.Country") == F.col("lkp_county"),
        "left"
    )
    .withColumn(
        "FIPS",
        F.when(
            F.col("main.Fips").isNull(),
            F.col("lkp_fips")
        ).otherwise(F.col("main.Fips"))
    )
    .drop("lkp_county", "lkp_fips")
)

display(silver_df)

Standlize the Data


Standardized text by converting the first character of each record to uppercase.

In [0]:
silver_df = standardize_string_columns(silver_df)

Removed numeric characters from first, middle, and last name fields using regular expressions.

In [0]:
silver_df = regex_replace_columns(
    silver_df,
    ["First", "Middle", "Last", "Maiden"],
    r"[0-9]"
)

In [0]:
display(silver_df)

Standardized gender values from 'M'/'F' to 'Male'/'Female'.

In [0]:
#Standalize the Gender column

silver_df = silver_df.withColumn(
    "Gender",

    F.when(
        F.col("Gender") == "M",
        "Male"
    ).when(
        F.col("Gender") == "F",
        "Female"
    ).otherwise(F.col("Gender"))
)

display(silver_df)

Standardized Martial values from 'M'/'S'/'D' to 'Marriage'/'Single'.

In [0]:
#Standlize the Martial Column

silver_df = silver_df.withColumn(
    "Marital",

    F.when(
        F.col("Marital") == "M",
        "Married"
    ).when(
        F.col("Marital") == "S",
        "Single"
    ).when(
        F.col("Marital") == "D",
        "Divorced"
    ).when(
        F.col("Marital") == "W",
        "Widowed"
    ).otherwise(F.col("Marital"))
)

display(silver_df)

In [0]:
silver_df = silver_df.withColumnRenamed(
    "Id",
    "Patient_id"
)
display(silver_df)

here concate the first,middel and last to full name

In [0]:

silver_df = silver_df.withColumn(
    "Name",

    F.concat_ws(
        " ",
        F.col("First"),
        F.col("Middle"),
        F.col("Last")
    )
)

display(silver_df)

%md
here performing null check,Duplicate Check,Primary Key Validation


In [0]:
validation_result = run_validations(
    silver_df,
    ["Patient_id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

only selecting the requried columns

In [0]:
silver_df = silver_df.select(
    "Patient_id",
    "Birthdate",
    "Deathdate",
    "Ssn",
    "Drivers",
    "Passport",
    "Prefix",
    "Name",
    "Suffix",
    "Maiden",
    "Marital",
    "Race",
    "Ethnicity",
    "Gender",
    "Birthplace",
    "Address",
    "City",
    "State",
    "Country",
    "FIPS",
    "Zip",
    "Lattitude",
    "Longtitude",
    "Healthcare_Expenses",
    "Healthcare_Coverage",
    "Income",
    "Filename"
)

In [0]:
#create tempview
silver_df.createOrReplaceTempView(
    "source_patient"
)


In [0]:
# silver_df.write.format("delta") \
#     .option("delta.enableChangeDataFeed", "true") \
#     .option("mergeSchema", "true") \
#     .mode("append") \
#     .saveAsTable(f"helathcare_silver.{silver_schema}.SL_patient")

In [0]:
%sql
select count(*) from helathcare_silver.helathcare_silver.sl_patient

In [0]:

%sql
MERGE INTO helathcare_silver.helathcare_silver.sl_patient tgt

USING source_patient src

ON tgt.patient_id = src.patient_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
patient_count = silver_df.count()

In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType
from datetime import datetime

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_silver.helathcare_silver.sl_patient"

# Get metadata
notebook_name, table_name, layer = get_audit_metadata(target_table)

status = "SUCCESS"
error_message = None
record_count = 0

In [0]:
# Duplicate Check

duplicate_check_status = (
    "PASS"
    if validation_result["duplicates"] == 0
    else "FAIL"
)

# Primary Key Check

primary_key_status = (
    validation_result["primary_key"]["status"]
)

# Null Check

null_count = (
    validation_result["nulls"]
    .agg(F.sum("null_count"))
    .collect()[0][0]
)

null_check_status = (
    "PASS"
    if null_count == 0
    else "FAIL"
)

# Standardization

standardization_status = "PASS"

In [0]:
end_time = datetime.now()

write_audit(
    target_table=target_table,
    run_id=run_id,
    record_count=patient_count,
    start_time=start_time,
    end_time=end_time,
    status=status,
    duplicate_check_status=duplicate_check_status,
    primary_key_status=primary_key_status,
    null_check_status=null_check_status,
    standardization_status=standardization_status,
    error_message=error_message
)